# BirdCLEF+ 2026 — Week 2 Training

## What's new vs Week 1
| Change | Why |
|---|---|
| All 5 folds trained | 5-model ensemble = biggest single LB gain |
| `train_soundscapes_labels.csv` added | Same domain as test — closes the domain gap |
| Focal loss replaces BCE | Handles class imbalance better for rare species |
| Stronger SpecAugment | More robust to soundscape noise |
| Secondary labels at 0.5 weight | Uses all available label information |
| 25 epochs with cosine LR | Longer training for all-fold runs |

**Settings:** Internet ON, GPU T4 x2, ∼6hr runtime

## 1. Setup

In [ ]:
import os, gc, math, random, warnings
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device :', DEVICE)
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

## 2. Configuration

In [ ]:
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────
    BASE_DIR         = Path('/kaggle/input/competitions/birdclef-2026')
    TRAIN_AUDIO_DIR  = BASE_DIR / 'train_audio'
    SOUNDSCAPE_DIR   = BASE_DIR / 'train_soundscapes'       # NEW Week 2
    TRAIN_CSV        = BASE_DIR / 'train.csv'
    SOUNDSCAPE_CSV   = BASE_DIR / 'train_soundscapes_labels.csv'  # NEW Week 2
    TAXONOMY_CSV     = BASE_DIR / 'taxonomy.csv'
    SAMPLE_SUB       = BASE_DIR / 'sample_submission.csv'
    OUTPUT_DIR       = Path('/kaggle/working')

    # ── Pretrained weights (same dataset as before) ────────────────────────
    # Keep internet ON so timm downloads weights automatically
    # OR attach the same pretrained weights dataset from Week 1
    WEIGHTS_PATH     = None   # Set to path string if using offline weights

    # ── Audio (unchanged from Week 1) ──────────────────────────────────────
    SAMPLE_RATE      = 32000
    WINDOW_SIZE      = 5
    AUDIO_LEN        = SAMPLE_RATE * WINDOW_SIZE

    # ── Mel spectrogram (unchanged) ────────────────────────────────────────
    N_FFT            = 1024
    HOP_LENGTH       = 64
    N_MELS           = 136
    FMIN             = 20
    FMAX             = 16000
    TARGET_SHAPE     = (256, 256)

    # ── Model (unchanged) ──────────────────────────────────────────────────
    MODEL_NAME       = 'efficientnet_b0'

    # ── Week 2 training changes ────────────────────────────────────────────
    N_FOLDS          = 5
    TRAIN_FOLDS      = [0, 1, 2, 3, 4]  # ALL folds (was [0] in Week 1)
    EPOCHS           = 25               # was 10
    BATCH_SIZE       = 32
    NUM_WORKERS      = 2
    LR               = 1e-3
    WEIGHT_DECAY     = 1e-4
    WARMUP_EPOCHS    = 2               # was 1
    MIN_LR           = 1e-6
    MIXUP_ALPHA      = 0.3             # was 0.15 — stronger

    # ── Soundscape label weights ───────────────────────────────────────────
    USE_SOUNDSCAPES  = True            # NEW: include train_soundscapes_labels
    SOUNDSCAPE_WEIGHT = 1.5            # upweight soundscape samples (same domain as test)

    # ── SpecAugment (stronger than Week 1) ────────────────────────────────
    FREQ_MASK_MAX    = 30              # was 20
    TIME_MASK_MAX    = 60              # was 40
    N_FREQ_MASKS     = 2
    N_TIME_MASKS     = 2

    SEED             = 42
    DEBUG            = False

cfg = CFG()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Soundscape dir exists:', cfg.SOUNDSCAPE_DIR.exists())
print('Soundscape CSV exists:', cfg.SOUNDSCAPE_CSV.exists())
print('Output dir           :', cfg.OUTPUT_DIR)

## 3. Load & Merge Data

Key change: merge `train.csv` with `train_soundscapes_labels.csv` into one unified DataFrame.

In [ ]:
# ── Load train.csv (clean clips) ──────────────────────────────────────────
train_df = pd.read_csv(cfg.TRAIN_CSV)
species_col = 'primary_label' if 'primary_label' in train_df.columns else 'species_code'
print(f'Train clips  : {len(train_df)} rows | species col: "{species_col}"')

# Build class list from train.csv (authoritative source)
all_classes = sorted(train_df[species_col].unique())
NUM_CLASSES  = len(all_classes)
le = LabelEncoder()
le.fit(all_classes)
class_list = le.classes_.tolist()
print(f'Classes      : {NUM_CLASSES}')

train_df['label_idx']    = le.transform(train_df[species_col])
train_df['source']       = 'clip'      # mark source
train_df['sample_weight'] = 1.0

# ── Load train_soundscapes_labels.csv ─────────────────────────────────────
if cfg.USE_SOUNDSCAPES and cfg.SOUNDSCAPE_CSV.exists():
    sl_df = pd.read_csv(cfg.SOUNDSCAPE_CSV)
    print(f'\nSoundscape labels CSV shape: {sl_df.shape}')
    print('Columns:', sl_df.columns.tolist())
    print(sl_df.head(3))
else:
    sl_df = None
    print('Soundscape CSV not found — using clip data only.')

In [ ]:
# ── Parse soundscape labels → same format as train_df ─────────────────────
# train_soundscapes_labels.csv format varies by year.
# Common format: filename | start_time | end_time | birds (space-separated species codes)
# We'll detect the format and normalise.

soundscape_rows = []

if sl_df is not None:
    cols = sl_df.columns.tolist()
    print('Detected columns:', cols)

    # Detect species column
    sp_col_sl = None
    for candidate in ['birds', 'primary_label', 'species_code', 'label', 'labels']:
        if candidate in cols:
            sp_col_sl = candidate
            break
    if sp_col_sl is None:
        sp_col_sl = cols[-1]   # fallback: last column
    print(f'Species column in soundscape CSV: "{sp_col_sl}"')

    # Detect filename column
    fn_col = None
    for candidate in ['filename', 'filepath', 'soundscape_id', 'recording_id']:
        if candidate in cols:
            fn_col = candidate
            break
    if fn_col is None:
        fn_col = cols[0]
    print(f'Filename column: "{fn_col}"')

    # Detect time columns
    start_col = 'start_time' if 'start_time' in cols else None
    end_col   = 'end_time'   if 'end_time'   in cols else None

    skipped = 0
    for _, row in sl_df.iterrows():
        species_raw = str(row[sp_col_sl]).strip()

        # Skip rows with no bird label
        if species_raw in ('', 'nan', 'nocall', 'no_call', 'unknown'):
            skipped += 1
            continue

        # May be space/comma-separated multiple species
        species_list = [s.strip() for s in species_raw.replace(',', ' ').split()]
        primary = species_list[0]

        # Only include species that exist in our class list
        if primary not in le.classes_:
            skipped += 1
            continue

        # Build filename path
        fname = str(row[fn_col])
        if not fname.endswith(('.ogg', '.wav', '.flac')):
            fname = fname + '.ogg'

        # Use start_time as offset into the soundscape file
        offset = float(row[start_col]) if start_col else 0.0

        secondary = [s for s in species_list[1:] if s in le.classes_]

        soundscape_rows.append({
            'filename'        : fname,
            species_col       : primary,
            'secondary_labels': ' '.join(secondary),
            'label_idx'       : le.transform([primary])[0],
            'source'          : 'soundscape',
            'sample_weight'   : cfg.SOUNDSCAPE_WEIGHT,
            '_offset'         : offset,
        })

    print(f'\nSoundscape rows parsed : {len(soundscape_rows)}')
    print(f'Skipped (no label/OOV) : {skipped}')
else:
    print('No soundscape data to parse.')

In [ ]:
# ── Merge clip + soundscape data ───────────────────────────────────────────
train_df['_offset'] = 0.0   # clips start at 0 (random offset handled in Dataset)

if soundscape_rows:
    sl_parsed = pd.DataFrame(soundscape_rows)
    combined_df = pd.concat([train_df, sl_parsed], ignore_index=True, sort=False)
else:
    combined_df = train_df.copy()

combined_df['sample_weight'] = combined_df['sample_weight'].fillna(1.0)
combined_df['_offset']       = combined_df['_offset'].fillna(0.0)
combined_df['source']        = combined_df['source'].fillna('clip')

print(f'Total training rows  : {len(combined_df)}')
print(combined_df['source'].value_counts().to_string())
print(f'\nClass distribution:')
counts = combined_df[species_col].value_counts()
print(f'  Min: {counts.min()}  |  Max: {counts.max()}  |  Median: {counts.median():.0f}')

## 4. Audio Pipeline

Same as Week 1 with one addition: tile short clips instead of zero-padding.

In [ ]:
def load_audio(filepath: str, sr: int, offset: float = 0.0,
               duration: float = 5.0) -> np.ndarray:
    """Load audio and tile (not pad) short clips — avoids silence artifacts."""
    target_len = int(sr * duration)
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=offset,
                             duration=duration, mono=True)
    except Exception:
        return np.zeros(target_len, dtype=np.float32)

    if len(y) == 0:
        return np.zeros(target_len, dtype=np.float32)

    # TILE instead of zero-pad (Week 2 improvement)
    if len(y) < target_len:
        reps = math.ceil(target_len / len(y))
        y = np.tile(y, reps)

    return y[:target_len].astype(np.float32)


def to_melspec(y: np.ndarray, cfg) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y, sr=cfg.SAMPLE_RATE, n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH, n_mels=cfg.N_MELS,
        fmin=cfg.FMIN, fmax=cfg.FMAX, power=2.0)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
    return mel.astype(np.float32)


def spec_augment(spec: np.ndarray, cfg) -> np.ndarray:
    spec = spec.copy()
    H, W = spec.shape
    for _ in range(cfg.N_FREQ_MASKS):
        f  = random.randint(0, cfg.FREQ_MASK_MAX)
        f0 = random.randint(0, max(0, H - f))
        spec[f0:f0+f, :] = 0.0
    for _ in range(cfg.N_TIME_MASKS):
        t  = random.randint(0, cfg.TIME_MASK_MAX)
        t0 = random.randint(0, max(0, W - t))
        spec[:, t0:t0+t] = 0.0
    return spec


def mixup(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam


def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1-lam) * criterion(pred, yb)


print('Audio pipeline defined.')

## 5. Dataset

In [ ]:
class BirdDataset(Dataset):
    """
    Unified dataset for both clip and soundscape rows.
    - Clips: random offset into the file
    - Soundscapes: fixed offset from _offset column (the labelled segment)
    """
    def __init__(self, df: pd.DataFrame, cfg, augment: bool = True):
        self.df      = df.reset_index(drop=True)
        self.cfg     = cfg
        self.augment = augment

    def __len__(self): return len(self.df)

    def _get_audio_dir(self, source, filename):
        if source == 'soundscape':
            return self.cfg.SOUNDSCAPE_DIR / filename
        return self.cfg.TRAIN_AUDIO_DIR / filename

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        source = row.get('source', 'clip')
        fp     = str(self._get_audio_dir(source, row['filename']))

        # Offset: random for clips, fixed for soundscape segments
        if source == 'soundscape':
            offset = float(row.get('_offset', 0.0))
        else:
            try:
                total = sf.info(fp).duration
            except Exception:
                total = self.cfg.WINDOW_SIZE
            max_off = max(0.0, total - self.cfg.WINDOW_SIZE)
            offset  = random.uniform(0, max_off) if self.augment and max_off > 0 else 0.0

        y    = load_audio(fp, self.cfg.SAMPLE_RATE, offset, self.cfg.WINDOW_SIZE)
        spec = to_melspec(y, self.cfg)
        if self.augment:
            spec = spec_augment(spec, self.cfg)
        spec = cv2.resize(spec, (self.cfg.TARGET_SHAPE[1], self.cfg.TARGET_SHAPE[0]),
                          interpolation=cv2.INTER_CUBIC)
        tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)

        # Label vector
        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        label[int(row['label_idx'])] = 1.0

        sec = str(row.get('secondary_labels', ''))
        if sec and sec != 'nan':
            for sp in sec.replace(',', ' ').split():
                sp = sp.strip()
                if sp in le.classes_:
                    label[le.transform([sp])[0]] = 0.5

        return tensor, label


print('Dataset defined.')

## 6. Model

Same EfficientNet-B0 + GeM architecture. No changes needed.

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(self.eps).pow(self.p), 1).pow(1/self.p)


class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes: int, cfg):
        super().__init__()

        if cfg.WEIGHTS_PATH and Path(cfg.WEIGHTS_PATH).exists():
            # Offline: load from dataset
            self.backbone = timm.create_model(
                cfg.MODEL_NAME, pretrained=False,
                num_classes=0, global_pool='', in_chans=1)
            sd  = torch.load(cfg.WEIGHTS_PATH, map_location='cpu', weights_only=True)
            key = 'conv_stem.weight'
            if key in sd:
                sd[key] = sd[key].mean(dim=1, keepdim=True)
            self.backbone.load_state_dict(sd, strict=False)
            print(f'Weights loaded from {cfg.WEIGHTS_PATH}')
        else:
            # Online: timm downloads automatically
            self.backbone = timm.create_model(
                cfg.MODEL_NAME, pretrained=True,
                num_classes=0, global_pool='', in_chans=1)
            print('Pretrained weights downloaded via timm.')

        d = self.backbone.num_features
        self.pool = GeM()
        self.bn   = nn.BatchNorm1d(d)
        self.drop = nn.Dropout(p=0.3)
        self.fc   = nn.Linear(d, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.drop(x)
        return self.fc(x)


# Quick test
m = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
out = m(torch.randn(2, 1, 256, 256).to(DEVICE))
print('Output shape:', out.shape)
print(f'Parameters  : {sum(p.numel() for p in m.parameters())/1e6:.1f}M')
del m; gc.collect(); torch.cuda.empty_cache()

## 7. Focal Loss

Replaces BCE. Focuses training on hard/rare examples — crucial for imbalanced species.

In [ ]:
class FocalLoss(nn.Module):
    """
    Binary Focal Loss for multilabel classification.
    gamma=2.0: standard focal loss — down-weights easy examples.
    alpha=0.25: prior probability for positives.

    When the model is confident and correct, the loss contribution is
    small. When the model is wrong on rare species, loss is amplified.
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce  = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none')
        prob = torch.sigmoid(logits)
        p_t  = targets * prob + (1 - targets) * (1 - prob)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


print('FocalLoss defined.')

## 8. Training Utilities

In [ ]:
def get_scheduler(optimizer, cfg, steps_per_epoch):
    total  = cfg.EPOCHS * steps_per_epoch
    warmup = cfg.WARMUP_EPOCHS * steps_per_epoch
    def lr_fn(step):
        if step < warmup:
            return step / max(1, warmup)
        prog = (step - warmup) / max(1, total - warmup)
        cos  = 0.5 * (1 + math.cos(math.pi * prog))
        return max(cfg.MIN_LR / cfg.LR, cos)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


def compute_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            try:
                aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else 0.0


def train_epoch(model, loader, optimizer, scheduler, criterion, cfg):
    model.train()
    losses = []
    for x, y in tqdm(loader, desc='  Train', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        if cfg.MIXUP_ALPHA > 0 and random.random() > 0.5:
            x, ya, yb, lam = mixup(x, y, cfg.MIXUP_ALPHA)
            loss = mixup_loss(criterion, model(x), ya, yb, lam)
        else:
            loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc='  Valid', leave=False):
        preds.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
        labels.append(y.numpy())
    preds  = np.concatenate(preds)
    labels = np.concatenate(labels)
    auc    = compute_auc((labels > 0.5).astype(int), preds)
    return auc, preds, labels


print('Training utilities defined.')

## 9. Cross-Validation Split

Stratified on primary label. Soundscape rows get assigned to folds proportionally.

In [ ]:
# Stratified split on clip data only (soundscape rows follow their clip fold)
clip_df  = combined_df[combined_df['source'] == 'clip'].copy()
sound_df = combined_df[combined_df['source'] == 'soundscape'].copy()

if cfg.DEBUG:
    clip_df  = clip_df.groupby(species_col).head(2).reset_index(drop=True)
    sound_df = sound_df.head(100).reset_index(drop=True)

skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
clip_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(clip_df, clip_df['label_idx'])):
    clip_df.loc[clip_df.index[val_idx], 'fold'] = fold

# Assign soundscape rows to folds round-robin
if len(sound_df) > 0:
    sound_df = sound_df.reset_index(drop=True)
    sound_df['fold'] = sound_df.index % cfg.N_FOLDS

combined_df = pd.concat([clip_df, sound_df], ignore_index=True)
print('Fold distribution (clip rows):')
print(clip_df['fold'].value_counts().sort_index().to_string())
if len(sound_df) > 0:
    print(f'\nSoundscape rows per fold: ~{len(sound_df)//cfg.N_FOLDS}')

## 10. Training Loop — All 5 Folds

In [ ]:
oof_preds  = np.zeros((len(combined_df), NUM_CLASSES), dtype=np.float32)
oof_labels = np.zeros((len(combined_df), NUM_CLASSES), dtype=np.float32)
best_aucs  = {}

criterion = FocalLoss(gamma=2.0, alpha=0.25)

for fold in cfg.TRAIN_FOLDS:
    print(f'\n{"="*55}')
    print(f'  FOLD {fold}')
    print(f'{"="*55}')

    trn_df = combined_df[combined_df['fold'] != fold].reset_index(drop=True)
    val_df = combined_df[combined_df['fold'] == fold].reset_index(drop=True)

    # Only validate on clip rows (clean labels for fair OOF)
    val_clip_df = val_df[val_df['source'] == 'clip'].reset_index(drop=True)

    print(f'  Train: {len(trn_df)}  |  Val (clips only): {len(val_clip_df)}')
    print(f'  Train soundscapes: {(trn_df["source"]=="soundscape").sum()}')

    trn_ds = BirdDataset(trn_df, cfg, augment=True)
    val_ds = BirdDataset(val_clip_df, cfg, augment=False)

    # Weighted sampler — upsample rare species and soundscape rows
    weights = trn_df['sample_weight'].values.astype(np.float32)
    sampler = WeightedRandomSampler(
        weights=weights, num_samples=len(weights), replacement=True)

    trn_loader = DataLoader(trn_ds, batch_size=cfg.BATCH_SIZE,
                            sampler=sampler, num_workers=cfg.NUM_WORKERS,
                            pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=cfg.NUM_WORKERS,
                            pin_memory=True)

    model     = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, cfg, len(trn_loader))

    best_auc  = 0.0
    best_path = cfg.OUTPUT_DIR / f'model_fold{fold}.pt'
    history   = []

    for epoch in range(1, cfg.EPOCHS + 1):
        trn_loss = train_epoch(model, trn_loader, optimizer, scheduler, criterion, cfg)
        val_auc, preds, labels = validate(model, val_loader)
        history.append({'epoch': epoch, 'loss': trn_loss, 'val_auc': val_auc})

        lr_now = scheduler.get_last_lr()[0]
        marker = ' ✔' if val_auc > best_auc else ''
        print(f'  E{epoch:02d}  loss={trn_loss:.4f}  auc={val_auc:.4f}  lr={lr_now:.2e}{marker}')

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)

    best_aucs[fold] = best_auc
    print(f'\n  Fold {fold} best AUC: {best_auc:.4f}')

    # OOF predictions with best model
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    val_idx = combined_df[combined_df['fold'] == fold].index
    val_all_ds  = BirdDataset(
        combined_df.loc[val_idx].reset_index(drop=True), cfg, augment=False)
    val_all_ldr = DataLoader(val_all_ds, batch_size=cfg.BATCH_SIZE,
                             shuffle=False, num_workers=cfg.NUM_WORKERS)
    _, oof_p, oof_l = validate(model, val_all_ldr)
    oof_preds[val_idx]  = oof_p
    oof_labels[val_idx] = (oof_l > 0.5).astype(np.float32)

    # Training curve
    hist = pd.DataFrame(history)
    fig, ax1 = plt.subplots(figsize=(9, 3))
    ax1.plot(hist['epoch'], hist['loss'], 'b-o', ms=4, label='Loss')
    ax1.set_ylabel('Focal Loss', color='b')
    ax2 = ax1.twinx()
    ax2.plot(hist['epoch'], hist['val_auc'], 'r-s', ms=4, label='Val AUC')
    ax2.set_ylabel('Val AUC', color='r')
    ax1.set_xlabel('Epoch')
    plt.title(f'Fold {fold}  —  best AUC {best_auc:.4f}')
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / f'curve_fold{fold}.png', dpi=100)
    plt.show()

    del model, trn_loader, val_loader, trn_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()

print(f'\nAll folds done.')
print('Best AUCs per fold:', {k: round(v,4) for k,v in best_aucs.items()})
print(f'Mean fold AUC     : {np.mean(list(best_aucs.values())):.4f}')

## 11. OOF Summary

In [ ]:
# OOF only on clip rows (clean labels)
clip_idx  = combined_df[combined_df['source'] == 'clip'].index
oof_auc   = compute_auc(oof_labels[clip_idx], oof_preds[clip_idx])
print(f'Overall OOF AUC (clip rows): {oof_auc:.4f}')

# Per-class AUC
per_class = []
for i in range(NUM_CLASSES):
    t = oof_labels[clip_idx, i]
    p = oof_preds[clip_idx, i]
    if t.sum() > 0:
        try: per_class.append(roc_auc_score(t, p))
        except: pass

plt.figure(figsize=(8, 3))
plt.hist(per_class, bins=30, color='steelblue', edgecolor='white')
plt.axvline(np.mean(per_class), color='red', linestyle='--',
            label=f'Mean = {np.mean(per_class):.3f}')
plt.xlabel('Per-class ROC-AUC')
plt.title('Per-class AUC (OOF)')
plt.legend(); plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'oof_per_class_auc.png', dpi=100)
plt.show()

print(f'Classes with AUC < 0.7: {sum(a < 0.7 for a in per_class)}')
print(f'Classes with AUC < 0.8: {sum(a < 0.8 for a in per_class)}')
print('\nSaved model files:')
for f in sorted(cfg.OUTPUT_DIR.glob('model_fold*.pt')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

---
## Done — Next Steps

1. **Publish outputs as a new Kaggle Dataset** — all 5 `model_fold*.pt` files
2. **Update inference notebook** — add all 5 model paths to `MODEL_PATHS`
3. **Submit** — 5-model ensemble will produce real predictions

### Update inference CFG MODEL_PATHS to:
```python
MODEL_PATHS = [
    '/kaggle/input/YOUR-DATASET/model_fold0.pt',
    '/kaggle/input/YOUR-DATASET/model_fold1.pt',
    '/kaggle/input/YOUR-DATASET/model_fold2.pt',
    '/kaggle/input/YOUR-DATASET/model_fold3.pt',
    '/kaggle/input/YOUR-DATASET/model_fold4.pt',
]
```

### Expected LB improvement
| Change | Expected gain |
|---|---|
| 5-fold ensemble | +0.03 to +0.05 |
| Soundscape training data | +0.05 to +0.08 |
| Focal loss + tiling | +0.01 to +0.02 |
| **Total expected** | **~0.75 to 0.82** |